# 01 - Collecte et Exploration des Données

## Objectif
Ce notebook documente la collecte, le chargement et l'exploration initiale des trois datasets de catastrophes maritimes :
1. **RMS Titanic** (1912) - 891 passagers
2. **RMS Lusitania** (1915) - 1961 personnes
3. **MV Sewol** (2014) - 476 personnes

## Sources des données
- **Titanic** : Kaggle Titanic Dataset (données publiques issues des archives historiques)
- **Lusitania** : Manifeste complet (RMS Lusitania Resource, archives historiques)
- **Sewol** : Dataset traduit du coréen (Korean Maritime Safety Tribunal reports)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ajouter le dossier parent au path pour importer src/
sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import (
    load_titanic, load_lusitania, load_sewol,
    get_data_quality_report
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Packages chargés avec succès.')

---
## 1. Chargement des Données Brutes

In [ ]:
# Chargement des trois datasets
titanic_raw = load_titanic()
lusitania_raw = load_lusitania()
sewol_raw = load_sewol()

print(f"Titanic  : {titanic_raw.shape[0]} lignes, {titanic_raw.shape[1]} colonnes")
print(f"Lusitania: {lusitania_raw.shape[0]} lignes, {lusitania_raw.shape[1]} colonnes")
print(f"Sewol    : {sewol_raw.shape[0]} lignes, {sewol_raw.shape[1]} colonnes")
print(f"\nTotal    : {titanic_raw.shape[0] + lusitania_raw.shape[0] + sewol_raw.shape[0]} individus")

---
## 2. Exploration du Dataset Titanic (1912)

Le RMS Titanic a coulé le 15 avril 1912 après avoir heurté un iceberg dans l'Atlantique Nord.
Sur environ 2224 personnes à bord, plus de 1500 ont péri. Ce dataset contient 891 passagers.

In [ ]:
print("=== TITANIC - Aperçu ===")
print(titanic_raw.head())
print("\n=== Types de données ===")
print(titanic_raw.dtypes)
print("\n=== Statistiques descriptives ===")
print(titanic_raw.describe())

In [ ]:
# Valeurs manquantes Titanic
missing_titanic = titanic_raw.isnull().sum()
missing_pct_titanic = (missing_titanic / len(titanic_raw) * 100).round(2)
missing_df = pd.DataFrame({'Manquantes': missing_titanic, 'Pourcentage (%)': missing_pct_titanic})
print("=== Valeurs manquantes Titanic ===")
print(missing_df[missing_df['Manquantes'] > 0])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploration du Dataset Titanic', fontsize=16, fontweight='bold')

# Distribution de survie
titanic_raw['Survived'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Distribution Survie')
axes[0, 0].set_xticklabels(['Décédé', 'Survécu'], rotation=0)

# Distribution par classe
titanic_raw['Pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution par Classe')
axes[0, 1].set_xticklabels(['1ère', '2ème', '3ème'], rotation=0)

# Distribution par sexe
titanic_raw['Sex'].value_counts().plot(kind='bar', ax=axes[0, 2], color=['#3498db', '#e91e8c'])
axes[0, 2].set_title('Distribution par Sexe')
axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=0)

# Distribution de l'âge
titanic_raw['Age'].dropna().hist(bins=30, ax=axes[1, 0], color='steelblue', edgecolor='black')
axes[1, 0].set_title('Distribution de l\'Âge')
axes[1, 0].set_xlabel('Âge')

# Survie par classe
titanic_raw.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Taux de Survie par Classe')
axes[1, 1].set_ylabel('Taux de survie')
axes[1, 1].set_xticklabels(['1ère', '2ème', '3ème'], rotation=0)

# Survie par sexe
titanic_raw.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1, 2], color=['#e91e8c', '#3498db'])
axes[1, 2].set_title('Taux de Survie par Sexe')
axes[1, 2].set_ylabel('Taux de survie')
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(f"\nTaux de survie global Titanic : {titanic_raw['Survived'].mean():.2%}")

---
## 3. Exploration du Dataset Lusitania (1915)

Le RMS Lusitania a été torpillé par un sous-marin allemand le 7 mai 1915 au large de l'Irlande.
Sur 1960 personnes à bord, 1198 ont péri (taux de mortalité ~61%).
Le naufrage a eu lieu en seulement 18 minutes, beaucoup plus rapide que le Titanic (~2h40).

In [ ]:
print("=== LUSITANIA - Aperçu ===")
print(lusitania_raw.head())
print("\n=== Types de données ===")
print(lusitania_raw.dtypes)
print("\n=== Valeurs de Fate ===")
print(lusitania_raw['Fate'].value_counts())

In [ ]:
# Valeurs manquantes Lusitania
missing_lusi = lusitania_raw.isnull().sum()
missing_pct_lusi = (missing_lusi / len(lusitania_raw) * 100).round(2)
missing_df_lusi = pd.DataFrame({'Manquantes': missing_lusi, 'Pourcentage (%)': missing_pct_lusi})
print("=== Valeurs manquantes Lusitania ===")
print(missing_df_lusi[missing_df_lusi['Manquantes'] > 0])

In [ ]:
# Filtrer les personnes réellement à bord
lusi_aboard = lusitania_raw[lusitania_raw['Fate'] != 'Not on board'].copy()
lusi_aboard['survived'] = lusi_aboard['Fate'].apply(lambda x: 1 if x == 'Saved' else 0)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploration du Dataset Lusitania', fontsize=16, fontweight='bold')

# Distribution de survie
lusi_aboard['survived'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Distribution Survie')
axes[0, 0].set_xticklabels(['Décédé', 'Survécu'], rotation=0)

# Distribution par département/classe
lusi_aboard['Department/Class'].value_counts().plot(kind='bar', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution Département/Classe')
axes[0, 1].tick_params(axis='x', rotation=45)

# Distribution par sexe
lusi_aboard['Sex'].value_counts().plot(kind='bar', ax=axes[0, 2], color=['#3498db', '#e91e8c'])
axes[0, 2].set_title('Distribution par Sexe')
axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=0)

# Distribution de l'âge
lusi_ages = pd.to_numeric(lusi_aboard['Age'].astype(str).str.replace('?', '', regex=False).str.strip(), errors='coerce')
lusi_ages.dropna().hist(bins=30, ax=axes[1, 0], color='steelblue', edgecolor='black')
axes[1, 0].set_title('Distribution de l\'Âge')
axes[1, 0].set_xlabel('Âge')

# Survie par classe/département
lusi_aboard.groupby('Department/Class')['survived'].mean().sort_values().plot(
    kind='barh', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Taux de Survie par Département/Classe')
axes[1, 1].set_xlabel('Taux de survie')

# Survie par sexe
lusi_aboard.groupby('Sex')['survived'].mean().plot(kind='bar', ax=axes[1, 2], color=['#e91e8c', '#3498db'])
axes[1, 2].set_title('Taux de Survie par Sexe')
axes[1, 2].set_ylabel('Taux de survie')
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(f"\nTaux de survie global Lusitania : {lusi_aboard['survived'].mean():.2%}")
print(f"Passenger/Crew : {lusi_aboard['Passenger/Crew'].value_counts().to_dict()}")
print(f"Citoyennetés : {lusi_aboard['Citizenship'].value_counts().head(10).to_dict()}")

---
## 4. Exploration du Dataset Sewol (2014)

Le MV Sewol a chaviré et coulé le 16 avril 2014 au large de la Corée du Sud.
Sur 476 personnes à bord, 304 ont péri, dont 250 lycéens en voyage scolaire.
C'est la catastrophe la plus récente (2014) et présente un contexte très différent des deux autres.

In [ ]:
print("=== SEWOL - Aperçu ===")
print(sewol_raw.head(10))
print("\n=== Types de données ===")
print(sewol_raw.dtypes)
print("\n=== Valeurs uniques des colonnes catégorielles ===")
for col in ['Category-1', 'Category-2', 'Raw', 'gender', 'location']:
    print(f"{col}: {sewol_raw[col].unique().tolist()}")

In [ ]:
# Valeurs manquantes Sewol
missing_sewol = sewol_raw.isnull().sum()
missing_pct_sewol = (missing_sewol / len(sewol_raw) * 100).round(2)
missing_df_sewol = pd.DataFrame({'Manquantes': missing_sewol, 'Pourcentage (%)': missing_pct_sewol})
print("=== Valeurs manquantes Sewol ===")
print(missing_df_sewol[missing_df_sewol['Manquantes'] > 0])

In [ ]:
sewol_raw['survived_binary'] = sewol_raw['Raw'].map({'survival': 1, 'Dead': 0})

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Exploration du Dataset Sewol', fontsize=16, fontweight='bold')

# Distribution de survie
sewol_raw['survived_binary'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Distribution Survie')
axes[0, 0].set_xticklabels(['Décédé', 'Survécu'], rotation=0)

# Distribution par catégorie
sewol_raw['Category-1'].value_counts().plot(kind='bar', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution Catégorie-1')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Distribution par sexe
sewol_raw['gender'].value_counts().plot(kind='bar', ax=axes[0, 2], color=['#3498db', '#e91e8c'])
axes[0, 2].set_title('Distribution par Genre')
axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=0)

# Distribution de l'âge
sewol_raw['age'].dropna().hist(bins=20, ax=axes[1, 0], color='steelblue', edgecolor='black')
axes[1, 0].set_title('Distribution de l\'Âge')
axes[1, 0].set_xlabel('Âge')

# Survie par catégorie
sewol_raw.groupby('Category-1')['survived_binary'].mean().plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Taux de Survie par Catégorie')
axes[1, 1].set_ylabel('Taux de survie')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

# Survie par sexe
sewol_raw.groupby('gender')['survived_binary'].mean().plot(kind='bar', ax=axes[1, 2], color=['#e91e8c', '#3498db'])
axes[1, 2].set_title('Taux de Survie par Genre')
axes[1, 2].set_ylabel('Taux de survie')
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print(f"\nTaux de survie global Sewol : {sewol_raw['survived_binary'].mean():.2%}")
print(f"Étage (floor) : {sewol_raw['floor'].value_counts().to_dict()}")
print(f"Localisation : {sewol_raw['location'].value_counts().to_dict()}")

---
## 5. Comparaison des Trois Datasets

In [ ]:
# Tableau comparatif
comparison = pd.DataFrame({
    'Catastrophe': ['Titanic', 'Lusitania', 'Sewol'],
    'Année': [1912, 1915, 2014],
    'Nombre d\'individus': [len(titanic_raw), len(lusi_aboard), len(sewol_raw)],
    'Taux de survie': [
        f"{titanic_raw['Survived'].mean():.2%}",
        f"{lusi_aboard['survived'].mean():.2%}",
        f"{sewol_raw['survived_binary'].mean():.2%}"
    ],
    'Colonnes': [titanic_raw.shape[1], lusitania_raw.shape[1], sewol_raw.shape[1]],
    'Type de catastrophe': ['Iceberg', 'Torpille', 'Chavirement'],
    'Durée du naufrage': ['~2h40', '~18min', '~2h30'],
    'Température eau': ['~-2°C', '~11°C', '~12°C']
})
print(comparison.to_string(index=False))

In [ ]:
# Comparaison des taux de survie
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparaison des Taux de Survie par Catastrophe', fontsize=14, fontweight='bold')

# Par sexe
survival_by_sex = pd.DataFrame({
    'Titanic': titanic_raw.groupby('Sex')['Survived'].mean(),
    'Lusitania': lusi_aboard.groupby('Sex')['survived'].mean(),
    'Sewol': sewol_raw.groupby('gender')['survived_binary'].mean()
})
survival_by_sex.plot(kind='bar', ax=axes[0], color=['#2c3e50', '#e74c3c', '#3498db'])
axes[0].set_title('Taux de survie par Sexe')
axes[0].set_ylabel('Taux de survie')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend()

# Distribution d'âge superposée
titanic_raw['Age'].dropna().hist(bins=30, ax=axes[1], alpha=0.5, label='Titanic', color='#2c3e50')
lusi_ages.dropna().hist(bins=30, ax=axes[1], alpha=0.5, label='Lusitania', color='#e74c3c')
sewol_raw['age'].dropna().hist(bins=20, ax=axes[1], alpha=0.5, label='Sewol', color='#3498db')
axes[1].set_title('Distribution d\'Âge Comparée')
axes[1].set_xlabel('Âge')
axes[1].legend()

# Taux de survie global
global_survival = pd.Series({
    'Titanic\n(1912)': titanic_raw['Survived'].mean(),
    'Lusitania\n(1915)': lusi_aboard['survived'].mean(),
    'Sewol\n(2014)': sewol_raw['survived_binary'].mean()
})
global_survival.plot(kind='bar', ax=axes[2], color=['#2c3e50', '#e74c3c', '#3498db'])
axes[2].set_title('Taux de Survie Global')
axes[2].set_ylabel('Taux de survie')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

---
## 6. Évaluation de la Qualité des Données

In [ ]:
# Rapport de qualité pour chaque dataset
for name, df in [('Titanic', titanic_raw), ('Lusitania', lusitania_raw), ('Sewol', sewol_raw)]:
    report = get_data_quality_report(df, name)
    print(f"\n{'='*60}")
    print(f"RAPPORT DE QUALITÉ : {report['name']}")
    print(f"{'='*60}")
    print(f"Dimensions : {report['shape']}")
    print(f"Doublons : {report['duplicates']}")
    print(f"\nValeurs manquantes :")
    for col, val in report['missing_values'].items():
        if val > 0:
            print(f"  {col}: {val} ({report['missing_pct'][col]}%)")

---
## 7. Réflexion Critique : Distribution Shift

### 3 problèmes de décalage de distribution entre 1912 et 2014

**1. Décalage démographique (Covariate Shift)**
- En 1912, les passagers du Titanic reflètent la société Edwardienne : forte stratification sociale, majorité d'immigrants européens en 3ème classe.
- Le Sewol en 2014 transportait principalement des lycéens coréens (âge ~17 ans, distribution très concentrée), ce qui crée une distribution d'âge radicalement différente.
- **Solution** : Pondération par densité de probabilité (importance weighting) ou normalisation par quantile pour chaque feature numérique.
- **Compromis** : La pondération peut créer une variance élevée si les distributions sont très différentes.

**2. Décalage conceptuel (Concept Drift)**
- La relation `sexe → survie` a changé : le Titanic appliquait strictement "femmes et enfants d'abord", tandis que le Sewol n'avait pas cette protocole (les élèves ont reçu l'ordre de rester dans leurs cabines).
- Les règles de sécurité maritime et les protocoles d'évacuation ont fondamentalement changé en 100 ans.
- **Solution** : Modélisation hiérarchique avec paramètres spécifiques par catastrophe, ou utilisation de features contextuelles (protocole d'évacuation, époque).
- **Compromis** : Ajouter des paramètres spécifiques réduit la capacité de généralisation.

**3. Décalage de mesure (Measurement Shift)**
- Les données de 1912 enregistrent la classe de billet, le tarif, la cabine. Le Sewol enregistre l'étage du navire et la position spatiale.
- Le concept de "classe sociale" est mesuré différemment : billets 1ère/2ème/3ème vs marin/étudiant/passager normal.
- L'âge est enregistré en années exactes pour le Titanic, parfois en fourchettes pour le Lusitania, et la plupart des Sewol sont identiques (17 ans).
- **Solution** : Créer des features abstraites unifiées (social_rank normalisé, age_group catégorique) plutôt que d'utiliser les mesures brutes.
- **Compromis** : La perte d'information spécifique lors de l'abstraction peut réduire la performance intra-catastrophe.

### Pourquoi le simple pooling produit de mauvais modèles

Le regroupement naïf de toutes les catastrophes suppose que la relation `features → survie` est identique partout. Or :
1. Les poids relatifs des features changent : le genre est le facteur dominant pour le Titanic ("women first"), mais l'emplacement sur le navire est dominant pour le Sewol.
2. Les bases de taux de survie diffèrent : ~38% pour le Titanic, ~39% pour le Lusitania, ~36% pour le Sewol.
3. Les interactions entre features sont spécifiques : classe×genre est très prédictif pour le Titanic, mais floor×location l'est pour le Sewol.
4. Le modèle poolé converge vers un compromis qui n'est optimal pour aucune catastrophe individuellement.

---
## 8. Considérations Éthiques

### Utilisation de données de victimes de catastrophes

- **Respect des victimes** : Ces données représentent des personnes réelles qui ont souffert et péri. Notre analyse se fait dans un cadre académique avec la gravité appropriée.
- **Données publiques** : Les manifestes du Titanic et du Lusitania sont des documents historiques publics. Les données du Sewol proviennent de rapports d'enquête officiels.
- **Anonymisation** : Bien que les noms soient présents dans les données historiques, nous ne les utilisons pas comme features prédictives.
- **Objectif** : L'analyse vise à comprendre les facteurs de survie pour améliorer la sécurité maritime future, pas pour juger les victimes.
- **Biais historique** : Les inégalités de survie reflètent les injustices de l'époque (discrimination de classe, de genre, de nationalité). Documenter ces biais sert un objectif éducatif.

---
## Résumé

| Aspect | Titanic | Lusitania | Sewol |
|--------|---------|-----------|-------|
| Taille | 891 | ~1960 | 476 |
| Qualité | Bonne (Age 20% manquant) | Moyenne (Age, notes incomplets) | Bonne (peu de manquants) |
| Features clés | Pclass, Sex, Age, Fare, SibSp/Parch | Dept/Class, Sex, Age, Citizenship | Category, gender, age, floor, location |
| Variable cible | Survived (0/1) | Fate (Saved/Lost) | Raw (survival/Dead) |
| Utilisable ? | Oui | Oui (après nettoyage) | Oui (après nettoyage) |

Les trois datasets sont exploitables et complémentaires. Le notebook suivant effectuera le nettoyage et l'unification.